# Scan Step Notebook (Drive Indexing)

## Goal
Show the scan pipeline step end-to-end and produce canonical scan indexes.

## Inputs
- Drive root folder id (`ROOT_ID`) in live mode
- Existing scan indexes in demo mode

## Outputs
- `scan/included.index.json`
- `scan/filtered.index.json`

## Run modes
- `live`: calls Google Drive and scans folders
- `demo`: loads pre-existing scan indexes and re-saves to canonical output paths

## Preconditions
- In `live` mode, auth/env for Drive access must be configured


## 1) Parameters


In [ ]:
from pathlib import Path

RUN_MODE = 'live'  # 'live' or 'demo'
ROOT_ID = '<drive_root_id>'
OUT_DIR = Path('scan')
INCLUDED_NAME = 'included.index.json'
FILTERED_NAME = 'filtered.index.json'
REPORT_NAME = 'scan_directory.report.json'
WORKERS = 6
VERBOSE = True

DEMO_INCLUDED_PATH = Path('scan/included.index.json')
DEMO_FILTERED_PATH = Path('scan/filtered.index.json')
DEMO_REPORT_PATH = Path('scan/scan_directory.report.json')

assert RUN_MODE in {'live', 'demo'}, "RUN_MODE must be either 'live' or 'demo'"
assert WORKERS >= 1, 'WORKERS must be >= 1'

OUT_DIR = Path(OUT_DIR)
included_path = OUT_DIR / INCLUDED_NAME
filtered_path = OUT_DIR / FILTERED_NAME
report_path = OUT_DIR / REPORT_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

included_path, filtered_path, report_path


## 2) Setup (Imports + Logging)


In [ ]:
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# VS Code/Jupyter may start from src/notebooks; make repo root importable for `src.*` modules.
def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Repository root not found (expected pyproject.toml and src/)')

repo_root = _find_repo_root(Path.cwd())
repo_root_str = str(repo_root)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

from src.drive_service import config
from src.drive_service.auth_service import load_creds
from src.drive_service.drive_client import get_drive_service, list_children
from src.drive_service.index import MapIndex
from src.drive_service.logging_utils import get_logger, setup_logging
from src.scan_directory.runtime import run_scan

setup_logging(VERBOSE)
logger = get_logger()
print(f'Repo root: {repo_root}')


## 3) Execute Scan (`live` or `demo`)


In [ ]:
reports = []
scan_errors = []
employees = []
total_included = 0
t0 = time.time()

if RUN_MODE == 'live':
    config.validate_env()
    if not ROOT_ID or ROOT_ID == "<drive_root_id>":
        raise ValueError('Set ROOT_ID to a real Drive folder id for live mode')

    creds = load_creds()
    drive = get_drive_service(creds)

    live_report = run_scan(
        creds=creds,
        drive=drive,
        root_id=ROOT_ID,
        workers=WORKERS,
        included_path=str(included_path),
        filtered_path=str(filtered_path),
        report_path=str(report_path),
        logger_obj=logger,
    )

    scan_errors = live_report.get("scan_errors", [])
    employees = [None] * live_report.get("employee_total", 0)

else:
    if not DEMO_INCLUDED_PATH.exists():
        raise FileNotFoundError(f"Demo included index not found: {DEMO_INCLUDED_PATH}")
    if not DEMO_FILTERED_PATH.exists():
        raise FileNotFoundError(f"Demo filtered index not found: {DEMO_FILTERED_PATH}")

    included_index = MapIndex.load_index(str(DEMO_INCLUDED_PATH), strict=True)
    filtered_index = MapIndex.load_index(str(DEMO_FILTERED_PATH), strict=True)

    included_index.save_index(str(included_path))
    filtered_index.save_index(str(filtered_path))

    demo_report = {
        "root_id": included_index.root_id,
        "employee_total": included_index.employee_count,
        "employee_succeeded": included_index.employee_count,
        "employee_failed": 0,
        "included_total": included_index.total_files,
        "filtered_total": filtered_index.total_files,
        "duration_seconds": round(time.time() - t0, 3),
        "scan_errors": [],
        "included_path": str(included_path),
        "filtered_path": str(filtered_path),
    }
    if DEMO_REPORT_PATH.exists():
        from src.drive_service.io_json import load_json
        demo_report = load_json(str(DEMO_REPORT_PATH))
    from src.drive_service.io_json import write_json
    write_json(str(report_path), demo_report)
    logger.info("Demo mode: indexes loaded and saved to canonical output paths")

loaded_included = MapIndex.load_index(str(included_path), strict=True)
loaded_filtered = MapIndex.load_index(str(filtered_path), strict=True)

from src.drive_service.io_json import load_json
scan_report = load_json(str(report_path))

summary = {
    "run_mode": RUN_MODE,
    "employee_folders": scan_report.get("employee_total", len(employees)),
    "included_total": loaded_included.total_files,
    "filtered_total": loaded_filtered.total_files,
    "scan_errors": len(scan_report.get("scan_errors", [])),
    "included_path": str(included_path),
    "filtered_path": str(filtered_path),
    "report_path": str(report_path),
}
summary


## 4) Validation Checks


In [ ]:
assert included_path.exists(), f"Missing output: {included_path}"
assert filtered_path.exists(), f"Missing output: {filtered_path}"
assert report_path.exists(), f"Missing output: {report_path}"

loaded_included = MapIndex.load_index(str(included_path), strict=True)
loaded_filtered = MapIndex.load_index(str(filtered_path), strict=True)

assert loaded_included.total_files == len(loaded_included.files), "Included total mismatch"
assert loaded_filtered.total_files == len(loaded_filtered.files), "Filtered total mismatch"

overlap = set(loaded_included.files).intersection(loaded_filtered.files)

print("Included path:", included_path)
print("Filtered path:", filtered_path)
print("Employee count:", loaded_included.employee_count)
print("Included total files:", loaded_included.total_files)
print("Filtered total files:", loaded_filtered.total_files)
print("Included/filtered key overlap (non-fatal):", len(overlap))

print("Report path:", report_path)


## 5) Diagnostics


In [ ]:
from collections import Counter

def as_record_list(index_obj):
    records = []
    for item in index_obj.files.values():
        if hasattr(item, "model_dump"):
            records.append(item.model_dump())
        else:
            records.append(dict(item))
    return records

included_records = as_record_list(loaded_included)
filtered_records = as_record_list(loaded_filtered)

reason_counter = Counter((r.get("reason") or "<missing>") for r in filtered_records)
type_counter = Counter((r.get("type") or "<missing>") for r in filtered_records)

print("Top filtered reasons:")
for reason, count in reason_counter.most_common(10):
    print(f"- {reason}: {count}")

print("\nFiltered type distribution:")
for typ, count in type_counter.most_common():
    print(f"- {typ}: {count}")

print("\nIncluded sample (10):")
for row in included_records[:10]:
    print({k: row.get(k) for k in ["employee", "file_name", "type", "drive_path"]})

print("\nFiltered sample (10):")
for row in filtered_records[:10]:
    print({k: row.get(k) for k in ["employee", "file_name", "type", "reason", "drive_path"]})

report_errors = scan_report.get("scan_errors", [])
if report_errors:
    print("\nEmployee scan errors:")
    for err in report_errors[:10]:
        print(err)


## 6) Next Step Handoff


In [ ]:
print("Next step: text extraction from the included index")
print()
print("python -m \"src.extract_text_from_index\" --index \"scan/included.index.json\" --out \"output/text_extracted\" --included \"included_text.index.json\" --excluded \"excluded_text.index.json\" --verbose")
print()
print("Notebook input path for next step:", included_path)
